**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB07R — Repaired Shared Sources and Regression Tests (added 2026-10-04)

**Track B · Repair notebook · Identity × Context project**

| Item | Content |
|---|---|
| **Purpose** | Write the four modules added by the 2026-10-04 repair (`scrub.py`, `steering.py`, `context_edits.py`, `graph.py`) and the regression tests (`tests/test_repairs.py`), then run every test against the **real** SEL-4L primary checkpoint. |
| **Why** | The review of MB01–MB13 (`NOTEBOOK_REVIEW_MB01_MB13.md`) found implementation defects in shared code (F01) and in notebook-local code (F02–F06, F15–F18). Every repaired behaviour has a test here; tests marked *documents the defect* pass only if the original code still shows the failure the repair addresses. |
| **Run order** | After MB07 (whose `%%writefile` cell now writes the repaired `circuits.py`) and MB02 (repaired `mi_metrics.py`); before any corrected section of MB03, MB10, MB11, MB11b, MB11c, MB13 or MB16. |
| **Outputs** | `src/scrub.py`, `src/steering.py`, `src/context_edits.py`, `src/graph.py`, `tests/test_repairs.py`, `results/MB07R_regression_tests.csv` |
| **Test data** | none |

The same test file also runs offline on a synthetic stand-in model (`tests/run_synthetic.py`); that run checks the code, this one checks it on the model under study.

In [1]:
import os
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /home/user/code/soccer_mi
Track A (read only): /home/user/code/soccer_mi/model_training_evaluation | exists: True
Track B: /home/user/code/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.patching as patching
import mi_src.circuits as circuits
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, patching, circuits]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Note: no GPU found. The full run works on CPU but is much slower.


Device: cpu | outputs: /home/user/code/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/scrub.py').exists(), "Missing dependency: src/scrub.py"

In [4]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/steering.py').exists(), "Missing dependency: src/steering.py"

In [5]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/context_edits.py').exists(), "Missing dependency: src/context_edits.py"

In [6]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/graph.py').exists(), "Missing dependency: src/graph.py"

In [7]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'tests/test_repairs.py').exists(), "Missing dependency: tests/test_repairs.py"

## Snapshot of every historical artifact, before any rerun

Re-running a repaired notebook from the top re-executes its original cells, which rewrite their historical files. With the
same code and data they rewrite the same content; on a GPU the last digits can differ. To make any change visible, the cell
below records the SHA-256 of every file under `results/`, `config/` and `artifacts/populations/` **once**
(`manifests/pre_repair_snapshot.json`; never overwritten). The last cell of this notebook, run again after all reruns,
lists every historical file whose bytes changed.

In [8]:
snapshot_path = OUT / "manifests" / "pre_repair_snapshot.json"
if snapshot_path.exists():
    print("pre-repair snapshot already recorded:", mi_setup.load_json(snapshot_path)["recorded_at"])
else:
    files = fingerprint.write_snapshot(snapshot_path, [OUT / "results", OUT / "config", OUT / "artifacts" / "populations"],
                                       patterns=("*.csv", "*.json", "*.parquet", "*.pt"))
    print(f"pre-repair snapshot written: {len(files)} files")

pre-repair snapshot written: 13 files


## Run the regression tests on SEL-4L (primary seed)

In [9]:
import importlib
try:
    import mi_src.scrub as scrub
    import mi_src.steering as steering
    import mi_src.context_edits as context_edits
    import mi_src.graph as graph
except ModuleNotFoundError as error:
    raise RuntimeError("BLOCKED: the repaired modules (scrub, steering, context_edits, graph) are missing. Run "
                       "MB07R_repaired_sources_and_regression_tests.ipynb, then restart the kernel.") from error
for module in [hooks, mi_metrics, circuits, scrub, steering, context_edits, graph]:
    importlib.reload(module)
for module in [circuits, mi_metrics, scrub, steering, context_edits, graph]:
    if getattr(module, "REPAIR_VERSION", None) != "2026-10-04":
        raise RuntimeError(f"BLOCKED: {module.__name__} on disk is not the repaired 2026-10-04 version. Re-run the repaired "
                           "%%writefile cells (MB02 for mi_metrics, MB07 for circuits, MB07R for the new modules), then restart.")
print("repaired modules loaded:", [m.__name__ for m in [circuits, mi_metrics, scrub, steering, context_edits, graph]])

verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
PRIMARY_SEED = int(prereg["model"]["primary_seed"])
run_directory = paths.rebase_run_dir(prereg["model"]["run_dirs"][str(PRIMARY_SEED)], TRACK["root"])
events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
val_conflicts = pd.read_parquet(OUT / "artifacts" / "populations" / "val_conflicts.parquet")
MODEL = mi_setup.load_model(run_directory, bundle)
p_player = val_conflicts[val_conflicts.population == "P-player"].reset_index(drop=True)
test_table = p_player[p_player.half == "estimation"].iloc[:64].reset_index(drop=True)
features = {name: tensor.cpu().numpy() for name, tensor in bundle["encoded"].tensors.items()}
context = {"model": MODEL, "bundle": bundle, "view": val_view, "table": test_table, "batch_size": 16,
           "features": features, "synthetic": False}

for name in list(sys.modules):
    if name == "tests" or name.startswith("tests."):
        del sys.modules[name]
(MI_DIR / "tests" / "__init__.py").touch()
if str(MI_DIR) not in sys.path:
    sys.path.insert(0, str(MI_DIR))
import tests.test_repairs as test_repairs
importlib.reload(test_repairs)
training.set_seed(0)
results = pd.DataFrame(test_repairs.run_all(context))
results.to_csv(OUT / "results" / "MB07R_regression_tests.csv", index=False)
fingerprint.record_files(MANIFEST, {"results/MB07R_regression_tests.csv": OUT / "results" / "MB07R_regression_tests.csv"})
pd.set_option("display.max_colwidth", 160)
print(results.to_string(index=False))
print(f"\n{int(results.passed.sum())} of {len(results)} passed on SEL-4L seed {PRIMARY_SEED} ({len(test_table)} estimation-half P-player events)")
if not results["passed"].all():
    raise RuntimeError("BLOCKED: a regression test failed on the real model; do not run any corrected section until it is understood.")

repaired modules loaded: ['mi_src.circuits', 'mi_src.mi_metrics', 'mi_src.scrub', 'mi_src.steering', 'mi_src.context_edits', 'mi_src.graph']


/home/user/code/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


                                                     test  passed                                                                                                                                                                                                                                                                                                                                                                                                                                                    detail
         test_receiver_gradients_match_finite_differences    True                                                                                                                                                                                                                                                                                                                                      29 receiver sites; largest relative error vs finite difference 3.24e-08; internal sites with non-zero

In [10]:
import tests.test_surgical_repairs as surgical_tests
surgical_results = pd.DataFrame(surgical_tests.run_all(context))
surgical_results.to_csv(OUT / 'results/MB07R_surgical_regression_tests.csv',index=False)
print(surgical_results.to_string(index=False))
assert surgical_results.passed.all(), 'Surgical regression failure'
fingerprint.record_files(MANIFEST,{'results/MB07R_surgical_regression_tests.csv':OUT/'results/MB07R_surgical_regression_tests.csv'})


                                         test  passed                                                                                                          detail
             R09 canonical graph control size    True                               Aliases preserve the graph; all 100 random controls match actual connection count
R10/R12 donor masks and coupled node controls    True Actual wrong-class donor mask matches every draw; changing one node preserves all other donors and branch modes
                R15 bounded continuous shifts    True                                     Support, clipped counts and boundary no-ops agree for both shift directions
                  R03 one-cluster uncertainty    True                                    One-match points remain descriptive; no artificial CI or inferential p-value


{'artifacts/conflict/conflict_set.parquet': {'path': 'artifacts/conflict/conflict_set.parquet',
  'bytes': 1735548,
  'sha256': '19d6197298f38803450ba3befbc0e1b744b6ab900eecb0f6374783c4cf2263b3',
  'recorded_at': '2026-10-05T17:22:56.743259+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_128_64_seed0.pt',
  'bytes': 330757,
  'sha256': '590fbc131586f5297fcd9ddfcdf7f2e1c503cf13e9f584142a5c31a10a79339c',
  'recorded_at': '2026-10-05T17:22:56.743711+00:00'},
 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt': {'path': 'artifacts/conflict/identity_audit/id_mlp_512_512_256_seed0.pt',
  'bytes': 2280753,
  'sha256': 'eab81928ab7743ddbe2cbdb1dc60d5b584408386e896705115145e7c8a6d62a2',
  'recorded_at': '2026-10-05T17:22:56.746435+00:00'},
 'artifacts/conflict/predictions/test_probs.parquet': {'path': 'artifacts/conflict/predictions/test_probs.parquet',
  'bytes': 7982855,
  'sha256': 'd21c146acd76ee17cb82670b

**What these tests establish.** That the repaired code does what it states on this model: gradients agree with finite
differences at every receiver (F01); the edge score is the limit of a small exact path patch; the scrub passes all three
self-checks and keeps an effect exactly when its abstraction is exactly true (F02, F03); the wrong-class control leaves
the scoring table untouched and preserves joint class sizes (F04); steering at β = 0 is a no-op and an event's push does
not depend on its batch (F05, F06); resampling never returns the original value when another exists and is batch-free
(F15); special codes are never edited (F16); code 0 of a context feature is a learned vector, so "set to 0" is not
removal, while the factor mean-ablation is exact (F17); the signed non-additivity is 0 for an additive response (F18);
the size-matched random-circuit space is counted correctly (F08); the edge-graph run reproduces the clean and corrupted
endpoints (F09).

**What they do not establish.** Anything about the scientific results. They test code paths, not hypotheses.

## After all reruns: which historical files changed?

Run this cell again after the repaired notebooks have been re-run. New `*_corrected` files are not listed (they are not in the
snapshot). A historical file listed here was rewritten with different bytes: compare it with the backup before relying on it.

In [11]:
changed = fingerprint.compare_snapshot(OUT / "manifests" / "pre_repair_snapshot.json")
print(f"historical files whose bytes changed since the pre-repair snapshot: {len(changed)}")
for item in changed:
    print(" ", item["reason"], item["path"])

historical files whose bytes changed since the pre-repair snapshot: 0
